## Data Exploration 

In [ ]:
import os
import glob

root = "data"
version = 'v2'
digital_path = "EN_x/"
record_low_path = "EN_y1/"
record_high_path = "EN_y2_headphone"

folders = [digital_path]

for folder in folders:
    files = glob.glob(os.path.join(root, version, folder, "*.wav"))
    
    files.sort()
    print(files)
    
    for i, file_path in enumerate(files, start=1):
        new_name = f"{i}.wav"
        new_path = os.path.join(root, version, folder, new_name)
        
        # os.rename(file_path, new_path)
    
    print(f"Renamed files in {folder}: {len(files)} files renamed.")


In [ ]:
import os
import glob
import numpy as np
import librosa
from IPython.display import Audio, display
from scipy.io import wavfile


root = "data"
version = 'v2'
digital_path = "EN_x/"
record_low_path = "EN_y1/"
record_high_path = "EN_y2_headphone/"

digital_files = glob.glob(os.path.join(root, version, digital_path, "*.wav"))
record_low_files = glob.glob(os.path.join(root, version, record_low_path, "*.wav"))
record_high_files = glob.glob(os.path.join(root, version, record_high_path, "*.wav"))

print(len(digital_files), len(record_low_files), len(record_high_files))


In [ ]:

idx = 11
digital_file = digital_files[idx]
record_low_file = digital_file.replace("EN_x/","EN_y1/")
record_high_file = digital_file.replace("EN_x/","EN_y2_headphone/")
print(digital_file, record_low_file, record_high_file)
digital_rate, digital_data = wavfile.read(digital_file)
record_low_rate, record_low_data = wavfile.read(record_low_file)
record_high_rate, record_high_data = wavfile.read(record_high_file)

In [ ]:
print("Playing Digital Signal:")
display(Audio(data=digital_data, rate=digital_rate))

print("Playing Recorded Low-Quality Signal:")
display(Audio(data=record_low_data, rate=record_low_rate))

print("Playing Recorded High-Quality Signal:")
display(Audio(data=record_high_data, rate=record_high_rate))

In [ ]:
import matplotlib.pyplot as plt

digital_time = [i / digital_rate for i in range(len(digital_data))]
record_low_time = [i / record_low_rate for i in range(len(record_low_data))]
record_high_time = [i / record_high_rate for i in range(len(record_high_data))]


fig, axs = plt.subplots(1, 3, figsize=(16, 4))

axs[0].plot(digital_time, digital_data, color='b')
axs[0].set_title('Digital Signal')
axs[0].set_xlabel('Time (s)')
axs[0].set_ylabel('Amplitude')

axs[1].plot(record_low_time, record_low_data, color='r')
axs[1].set_title('Recorded Low-Quality Signal')
axs[1].set_xlabel('Time (s)')
axs[1].set_ylabel('Amplitude')

axs[2].plot(record_high_time, record_high_data, color='g')
axs[2].set_title('Recorded High-Quality Signal')
axs[2].set_xlabel('Time (s)')
axs[2].set_ylabel('Amplitude')

plt.tight_layout()
plt.show()

In [ ]:
from scipy.signal import stft

duration_sec = 10

digital_samples = int(duration_sec * digital_rate)
record_low_samples = int(duration_sec * record_low_rate)
record_high_samples = int(duration_sec * record_high_rate)

digital_data = digital_data[:digital_samples]
record_low_data = record_low_data[:record_low_samples]
record_high_data = record_high_data[:record_high_samples]

f1, t1, Zxx1 = stft(digital_data, fs=digital_rate, nperseg=1024)
f2, t2, Zxx2 = stft(record_low_data, fs=record_low_rate, nperseg=1024)
f3, t3, Zxx3 = stft(record_high_data, fs=record_high_rate, nperseg=1024)


fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(21, 6))

ax1.pcolormesh(t1, f1, np.abs(Zxx1), shading='gouraud')
ax1.set_title('Digital Signal Spectrogram (First 10 Seconds)')
ax1.set_xlabel('Time [sec]')
ax1.set_ylabel('Frequency [Hz]')
ax1.set_ylim([0, digital_rate / 2])  

ax2.pcolormesh(t2, f2, np.abs(Zxx2), shading='gouraud')
ax2.set_title('Recorded Low-Quality Signal Spectrogram (First 10 Seconds)')
ax2.set_xlabel('Time [sec]')
ax2.set_ylabel('Frequency [Hz]')
ax2.set_ylim([0, record_low_rate / 2])  

ax3.pcolormesh(t3, f3, np.abs(Zxx3), shading='gouraud')
ax3.set_title('Recorded High-Quality Signal Spectrogram (First 10 Seconds)')
ax3.set_xlabel('Time [sec]')
ax3.set_ylabel('Frequency [Hz]')
ax3.set_ylim([0, record_high_rate / 2])  

plt.tight_layout()
plt.show()



In [ ]:
import os
import librosa
from scipy.io import wavfile
from tqdm import tqdm


def preprocess(digital_path, record_low_path, segment_length=5, stride_length=0.5, target_sampling_rate=16000):
    digital_waveforms = []
    record_low_waveforms = []

    for i in tqdm(range(1, 31)):
        digital_file = os.path.join(digital_path, f"{i}.wav")
        record_low_file = os.path.join(record_low_path, f"{i}.wav")
        
        digital_data, _ = librosa.load(digital_file, sr=target_sampling_rate)
        record_low_data, _ = librosa.load(record_low_file, sr=target_sampling_rate)
        
        segment_samples = int(segment_length * target_sampling_rate)
        stride_samples = int(stride_length * target_sampling_rate)
        
        for start in range(0, len(digital_data) - segment_samples + 1, stride_samples):
            digital_segment = digital_data[start:start + segment_samples]
            record_low_segment = record_low_data[start:start + segment_samples]
            
            digital_waveforms.append(digital_segment)
            record_low_waveforms.append(record_low_segment)
    
    return digital_waveforms, record_low_waveforms


digital_waveforms, record_high_waveforms = preprocess(f"data/{version}/EN_x", f"data/{version}/EN_y2_headphone")

In [ ]:
print(len(digital_waveforms))

In [ ]:
import random

sample_index = random.randint(0, len(digital_waveforms) - 1)

digital_sample = digital_waveforms[sample_index]
record_high_sample = record_high_waveforms[sample_index]

plt.figure(figsize=(12, 5))

plt.subplot(2, 1, 1)
plt.plot(digital_sample)
plt.title("Digital Sample")
plt.xlabel("Time (samples)")
plt.ylabel("Amplitude")

plt.subplot(2, 1, 2)
plt.plot(record_high_sample)
plt.title("Record Low Sample")
plt.xlabel("Time (samples)")
plt.ylabel("Amplitude")

plt.tight_layout()
plt.show()


In [ ]:
print("Playing Digital Sample:")
display(Audio(digital_sample, rate=16000))

print("Playing Record Sample:")
display(Audio(record_high_sample, rate=16000))

## Data Loader

In [12]:
BATCH_SIZE = 2
NUM_WORKERS = 2
SHUFFLE = True
SAMPLE_RATE = 44100  
SEGMENT_LENGTH = 5
STRIDE_LENGTH = 0.5
STAGE = 1

In [13]:
import os
import librosa
from tqdm import tqdm


def preprocess(digital_path, record_low_path, segment_length=SEGMENT_LENGTH, stride_length=STRIDE_LENGTH, target_sampling_rate=SAMPLE_RATE, total_files=30):
    test_files = 5
    train_val_files = total_files - test_files

    train_end = int(0.8 * train_val_files)
    val_end = train_end + int(0.2 * train_val_files)

    train_digital_waveforms, val_digital_waveforms, test_digital_waveforms = [], [], []
    train_record_low_waveforms, val_record_low_waveforms, test_record_low_waveforms = [], [], []

    for i in tqdm(range(1, total_files + 1)):
        if i <= test_files:
            digital_waveforms = test_digital_waveforms
            record_low_waveforms = test_record_low_waveforms
        elif i <= train_end + test_files:
            digital_waveforms = train_digital_waveforms
            record_low_waveforms = train_record_low_waveforms
        elif i <= val_end + test_files:
            digital_waveforms = val_digital_waveforms
            record_low_waveforms = val_record_low_waveforms

        digital_file = os.path.join(digital_path, f"{i}.wav")
        record_low_file = os.path.join(record_low_path, f"{i}.wav")

        digital_data, _ = librosa.load(digital_file, sr=target_sampling_rate)
        record_low_data, _ = librosa.load(record_low_file, sr=target_sampling_rate)

        segment_samples = int(segment_length * target_sampling_rate)
        stride_samples = int(stride_length * target_sampling_rate)

        for start in range(0, len(digital_data) - segment_samples + 1, stride_samples):
            digital_segment = digital_data[start:start + segment_samples]
            record_low_segment = record_low_data[start:start + segment_samples]

            digital_waveforms.append(digital_segment)
            record_low_waveforms.append(record_low_segment)

    return (train_digital_waveforms, train_record_low_waveforms), \
           (val_digital_waveforms, val_record_low_waveforms), \
           (test_digital_waveforms, test_record_low_waveforms)


In [ ]:
version = 'v2'
STAGE = 2

(train_digital_waveforms, train_record_low_waveforms), \
(val_digital_waveforms, val_record_low_waveforms), \
(test_digital_waveforms, test_record_low_waveforms) = preprocess(f"data/{version}/EN_x", f"data/{version}/EN_y{STAGE}_headphone")
print(len(train_digital_waveforms), len(val_digital_waveforms), len(test_digital_waveforms))


In [15]:
import torch
from torch.utils.data import Dataset, DataLoader

class EqualizerDataset(Dataset):
    def __init__(self, digital_waveforms, record_low_waveforms, return_dict=False):
        assert len(digital_waveforms) == len(record_low_waveforms), \
            "Input and output waveforms lists must be of the same length"

        self.digital_waveforms = digital_waveforms
        self.record_low_waveforms = record_low_waveforms
        self.return_dict = return_dict

    def __len__(self):
        return len(self.digital_waveforms)

    def __getitem__(self, idx):
        digital_sample = torch.tensor(self.digital_waveforms[idx], dtype=torch.float32)
        record_low_sample = torch.tensor(self.record_low_waveforms[idx], dtype=torch.float32)
        if self.return_dict:
            return {'input_values': digital_sample, 'labels': record_low_sample}

        return digital_sample, record_low_sample

In [16]:
train_dataset = EqualizerDataset(train_digital_waveforms, train_record_low_waveforms)
val_dataset = EqualizerDataset(val_digital_waveforms, val_record_low_waveforms)
test_dataset = EqualizerDataset(test_digital_waveforms, test_record_low_waveforms)

In [ ]:
for digital_batch, record_low_batch in tqdm(train_dataset):
    print("Digital Batch Shape:", digital_batch.shape)
    print("Record Low Batch Shape:", record_low_batch.shape)
    break


## Model

### Wav2Vec Encoder

In [ ]:
import torch
from transformers import Wav2Vec2Model

wav2vec_model = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base")

sample_waveform = torch.tensor(digital_batch).unsqueeze(0) 

print("Sample shape before forward pass:", sample_waveform.shape)

with torch.no_grad():
    wav2vec_output = wav2vec_model(sample_waveform)

print("Output shape from wav2vec 2.0:", wav2vec_output.last_hidden_state.shape)


In [ ]:
import torch
import torch.nn as nn
from transformers import Wav2Vec2Model


class BLSTM(nn.Module):
    def __init__(self, dim, layers=2, bi=True):
        super().__init__()
        self.lstm = nn.LSTM(input_size=dim, hidden_size=dim, num_layers=layers, bidirectional=bi, batch_first=True)
        self.linear = nn.Linear(dim * 2, dim) if bi else None

    def forward(self, x):
        x, _ = self.lstm(x)  
        if self.linear:
            x = self.linear(x)
        return x  


class Decoder(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.down_conv1 = nn.Conv1d(input_dim, hidden_dim, kernel_size=4, stride=2, padding=1)
        self.down_conv2 = nn.Conv1d(hidden_dim, hidden_dim * 2, kernel_size=4, stride=2, padding=1)
        self.down_conv3 = nn.Conv1d(hidden_dim * 2, hidden_dim * 4, kernel_size=4, stride=2, padding=1)
        
        self.up_conv1 = nn.ConvTranspose1d(hidden_dim * 4, hidden_dim * 2, kernel_size=4, stride=2, padding=1)
        self.up_conv2 = nn.ConvTranspose1d(hidden_dim * 2, hidden_dim, kernel_size=4, stride=2, padding=1)
        self.final_conv = nn.ConvTranspose1d(hidden_dim, 1, kernel_size=4, stride=2, padding=1)
        
    def forward(self, x):
        x1 = self.down_conv1(x) 
        x2 = self.down_conv2(x1)  
        x3 = self.down_conv3(x2)
        
        x = self.up_conv1(x3) 
        x = self.up_conv2(x) 
        x = self.final_conv(x)     
        
        return x


class Wav2VecEqualizer(nn.Module):
    def __init__(self, freeze_encoder=True, target_length=160000):
        super().__init__()
        
        self.encoder = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base")

        if freeze_encoder:
            for name, param in self.encoder.named_parameters():
                if "pos_conv_embed" in name:
                    param.requires_grad = True
                else:
                    param.requires_grad = False

        # self.lstm = BLSTM(dim=768)
        
        self.decoder = Decoder(input_dim=499, hidden_dim=256)
        self.target_length = target_length
        self.fc = nn.Linear(768, target_length)

    def forward(self, x):
        x = self.encoder(x).last_hidden_state
        x = self.decoder(x).squeeze(1)        
        x = self.fc(x)
        return x

    

In [ ]:
model = Wav2VecEqualizer(freeze_encoder=True)
sample_input = torch.randn(2, 160000)  
output = model(sample_input)
print("Output shape:", output.shape)

### Demuc

In [ ]:
from IPython import display as disp
import torch
import torchaudio
from denoiser import pretrained
from denoiser.dsp import convert_audio
import librosa

model = pretrained.dns64().cpu()
wav, _ = librosa.load("data/EN_x/0.wav", sr=model.sample_rate)
with torch.no_grad():
    denoised = model(torch.FloatTensor(wav[None]))[0]
disp.display(disp.Audio(wav.data, rate=model.sample_rate))
disp.display(disp.Audio(denoised.data, rate=model.sample_rate))

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters: {total_params}")


### Diffusion

In [ ]:
from audio_diffusion_pytorch import DiffusionModel, UNetV0, VDiffusion, VSampler, VInpainter
import torch
from IPython.display import Audio

import soundfile as sf

model = DiffusionModel(
    net_t=UNetV0,
    in_channels=1,
    channels=[8, 32, 64, 128, 256, 512], # U-Net: channels at each layer
    factors=[1, 4, 4, 4, 2, 2], # U-Net: downsampling and upsampling factors at each layer
    items=[1, 2, 2, 2, 2, 2], # U-Net: number of repeating items at each layer
    attentions=[0, 0, 0, 0, 0, 1], # U-Net: attention enabled/disabled at each layer
    attention_heads=8,
    attention_features=64,
    diffusion_t=VDiffusion,
    sampler_t=VSampler,
    use_text_conditioning=True,
    use_embedding_cfg=True,
    embedding_max_length=64,
    embedding_features=768,
    cross_attentions=[0, 0, 0, 1, 1, 1], # U-Net: cross-attention enabled/disabled at each layer
)

model.load_state_dict(torch.load("assets/best_model_checkpoint.pth", map_location='cpu'))
model.eval()

noise = torch.randn(1, 1, 2**16)
print(noise)

In [ ]:
sample = model.sample(
    noise,
    text=["1"],
    embedding_scale=15.0,
    num_steps=100,
    show_progress=True
)
sf.write("low.wav", sample.squeeze().cpu().numpy(), samplerate=16000)
print("Generated low quality music saved.")
Audio("low.wav", rate=16000)

In [ ]:
# noise = torch.randn(1, 1, 2**16)
sample = model.sample(
    noise,
    text=["3"],
    embedding_scale=12.0,
    num_steps=100,
    show_progress=True

)
sf.write("high.wav", sample.squeeze().cpu().numpy(), samplerate=16000)
print("Generated high quality music saved.")
Audio("high.wav", rate=16000)


In [ ]:
sample = model.sample(
    noise,
    text=["2"],
    embedding_scale=12.0,
    num_steps=50,
    show_progress=True

)
sf.write("normal.wav", sample.squeeze().cpu().numpy(), samplerate=16000)
print("Generated normal quality music saved.")
Audio("normal.wav", rate=16000)

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")


In [ ]:
from diffusers import AudioLDM2Pipeline
import torch

repo_id = "cvssp/audioldm2-music"
# repo_id = "cvssp/audioldm2"
pipe = AudioLDM2Pipeline.from_pretrained(repo_id)
pipe = pipe.to("cpu")


In [ ]:
prompt = '''  Generate high-quality audio with an upbeat rhythm and a dance groove.'''
# negative_prompt = "Low quality."

generator = torch.Generator("cpu").manual_seed(0)

audio = pipe(
    prompt,
    # negative_prompt=negative_prompt,
    num_inference_steps=5,
    audio_length_in_s=5.0,
    num_waveforms_per_prompt=3,
).audios

In [ ]:
import soundfile as sf
from IPython.display import Audio

sf.write("ex4.wav", audio[0], samplerate=16000)
Audio("ex4.wav", rate=16000)

## Trainer

In [ ]:
from transformers import Trainer, TrainingArguments
import torch
from torch.utils.data import Dataset
import torch.nn as nn
from loss import STFTLoss


stft_loss_fn = STFTLoss()

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions
    mse = ((preds - labels) ** 2).mean()
    stft_loss = stft_loss_fn(preds, labels)
    
    return {"mse": mse, "stft_loss": stft_loss}


class TrainerModelWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.mse_loss_fn = nn.MSELoss()  

    def forward(self, input_values, labels=None):
        outputs = self.model(input_values)
        loss = self.mse_loss_fn(outputs, labels) + stft_loss_fn(outputs, labels) if labels is not None else None
        
        return (loss, outputs) if loss is not None else outputs


In [ ]:
training_args = TrainingArguments(
    output_dir="assets",
    eval_strategy="epoch",
    learning_rate=5e-5,
    save_strategy='epoch',
    logging_steps=4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=10,
    weight_decay=0.01,
    logging_dir="assets/logs",
    report_to="wandb",
    save_safetensors=False,
)

model = TrainerModelWrapper(Wav2VecEqualizer(freeze_encoder=False))

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

# trainer.train()

### Stage 2

In [ ]:
import torch
from IPython.display import Audio
import random
from safetensors.torch import load_file
from models.demucs_equalizer import DemucsEqualizer, DoubleDemucsEqualizer

STAGE = 2

(train_digital_waveforms, train_record_low_waveforms), \
(val_digital_waveforms, val_record_low_waveforms), \
(test_digital_waveforms, test_record_low_waveforms) = preprocess("data/EN_x", f"data/EN_y{STAGE}_earphone")

train_dataset = EqualizerDataset(train_digital_waveforms, train_record_low_waveforms, return_dict=True)
val_dataset = EqualizerDataset(val_digital_waveforms, val_record_low_waveforms, return_dict=True)
test_dataset = EqualizerDataset(test_digital_waveforms, test_record_low_waveforms, return_dict=True)


In [ ]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

checkpoint_path = "assets/stage2-earphone/pytorch_model.bin" # assets/stage2-headphone/pytorch_model.bin

model = DoubleDemucsEqualizer("assets/stage1/pytorch_model.bin", device=device)
state_dict = torch.load(checkpoint_path, map_location=device)
state_dict = {k[6:]: v for k, v in state_dict.items()}

model.load_state_dict(state_dict)
model.eval()  

model = model.to(device)

sample_idx = random.randint(500, 700)
print(sample_idx) # 767 586 598
sample = test_dataset[sample_idx]
y1_sample = test_dataset_stage_1[sample_idx]

input_waveform = sample['input_values'].unsqueeze(0).unsqueeze(0)
target_waveform = sample['labels'].numpy()  

with torch.no_grad():
    first_waveform = model.model2(input_waveform).squeeze(0)
    second_waveform = model.model1(first_waveform).squeeze(0).squeeze(0).cpu().numpy()

print("Original Digital Audio:")
display(Audio(input_waveform.squeeze().numpy(), rate=16000)) 

print("Y1 Label:")
display(Audio(y1_sample['labels'].reshape(-1), rate=16000)) 

print("Target Audio:")
display(Audio(target_waveform, rate=16000))

print("First Audio:")
display(Audio(first_waveform, rate=16000))

print("Second Audio:")
display(Audio(second_waveform, rate=16000))


## Evaluation

### Stage 1

In [ ]:
import torch
from IPython.display import Audio
import random
from safetensors.torch import load_file
from models.demucs_equalizer import DemucsEqualizer
from safetensors.torch import load_file
from utils import preprocess 
from dataset import EqualizerDataset

STAGE = 1
version = 'v2'

(train_digital_waveforms, train_record_low_waveforms), \
(val_digital_waveforms, val_record_low_waveforms), \
(test_digital_waveforms, test_record_low_waveforms) = preprocess(f"data/{version}/EN_x", f"data/{version}/EN_y{STAGE}", target_sampling_rate=44100, total_files=15)

test_dataset_stage_1 = EqualizerDataset(test_digital_waveforms, test_record_low_waveforms, speakers=[None] * len(test_record_low_waveforms), return_dict=True)


In [ ]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

checkpoint_path = "assets/v2/15_stage1/pytorch_model.bin"

model = DemucsEqualizer(device=device)
state_dict = torch.load(checkpoint_path, map_location=device)
state_dict = {k[6:].replace("model",'model1'): v for k, v in state_dict.items()}
model.load_state_dict(state_dict)
model.eval()  

model = model.to(device)

sample_idx = random.randint(0, len(test_dataset_stage_1) - 1) # 626 7 747 54 793 568 1077 856
print(sample_idx)
sample = test_dataset_stage_1[sample_idx]

input_waveform = sample['input_values'].unsqueeze(0).unsqueeze(0)
target_waveform = sample['labels'].numpy()  

with torch.no_grad():
    enhanced_waveform = model(input_waveform).squeeze(0).squeeze(0).cpu().numpy()

print("Original Digital Audio:")
display(Audio(input_waveform.squeeze().numpy(), rate=44100)) 

print("Target Audio:")
display(Audio(target_waveform, rate=44100))

print("Output Audio:")
display(Audio(enhanced_waveform, rate=44100))


### Stage 2

In [ ]:
import torch
from IPython.display import Audio
import random
from safetensors.torch import load_file
from models.demucs_equalizer import DemucsEqualizer, DoubleDemucsEqualizer

STAGE = 2
VERSION = 'v2' 

(train_digital_waveforms, train_record_low_waveforms), \
(val_digital_waveforms, val_record_low_waveforms), \
(test_digital_waveforms, test_record_low_waveforms) = preprocess(f"data/{VERSION}/EN_x", f"data/{VERSION}/EN_y{STAGE}_headphone", target_sampling_rate=44100)

train_dataset = EqualizerDataset(train_digital_waveforms, train_record_low_waveforms, return_dict=True)
val_dataset = EqualizerDataset(val_digital_waveforms, val_record_low_waveforms, return_dict=True)
test_dataset = EqualizerDataset(test_digital_waveforms, test_record_low_waveforms, return_dict=True)


In [ ]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

checkpoint_path = "assets/v2/60_stage2/pytorch_model.bin" # assets/stage2-headphone/pytorch_model.bin

model = DoubleDemucsEqualizer("assets/v2/60_stage1/pytorch_model.bin", device=device)
state_dict = torch.load(checkpoint_path, map_location=device)
state_dict = {k[6:]: v for k, v in state_dict.items()}
model.load_state_dict(state_dict)
model.eval()  

model = model.to(device)

sample_idx = random.randint(0, len(test_dataset_stage_1) - 1) # 87 296 625
print(sample_idx) 
sample = test_dataset[sample_idx]
y1_sample = test_dataset_stage_1[sample_idx]

input_waveform = sample['input_values'].unsqueeze(0)
if len(input_waveform.shape) == 2:
    input_waveform = input_waveform.unsqueeze(1)
target_waveform = sample['labels'].numpy()  
input_waveform = torch.cat([input_waveform, input_waveform], dim=1)        

with torch.no_grad():
    first_waveform = model.model2(input_waveform)
    first_waveform = first_waveform.mean(dim=1)[:,0,:]  
    first_waveform = first_waveform.reshape(first_waveform.shape[0], -1)
    second_waveform = model.model1(first_waveform).squeeze(0).cpu().numpy()

print("Original Digital Audio:")
display(Audio(input_waveform.squeeze().numpy(), rate=44100)) 

print("Y1 Label:")
display(Audio(y1_sample['labels'].reshape(-1), rate=44100)) 

print("Target Audio:")
display(Audio(target_waveform, rate=44100))

print("First Audio:")
display(Audio(first_waveform, rate=44100))

print("Second Audio:")
display(Audio(second_waveform, rate=44100))


### Style Transfer

In [ ]:
import torch
from IPython.display import Audio
import random
from safetensors.torch import load_file
from models.demucs_equalizer import StyleTransform
from utils import preprocess 
from dataset import EqualizerDataset

STAGE = 1
version = 'v2'

(train_digital_waveforms, train_record_low_waveforms), \
(val_digital_waveforms, val_record_low_waveforms), \
(test_digital_waveforms, test_record_low_waveforms) = preprocess(f"data/{version}/EN_x", f"data/{version}/EN_y1", total_files=15, target_sampling_rate=44100)

test_dataset_y1 = EqualizerDataset(test_digital_waveforms, test_record_low_waveforms, speakers=["y1"]*len(test_record_low_waveforms), return_dict=True)


(train_digital_waveforms, train_record_high_waveforms), \
(val_digital_waveforms, val_record_high_waveforms), \
(test_digital_waveforms, test_record_high_waveforms) = preprocess(f"data/{version}/EN_x", f"data/{version}/EN_y2_headphone", total_files=15, target_sampling_rate=44100)

test_dataset_y2 = EqualizerDataset(test_digital_waveforms, test_record_high_waveforms, speakers=["y2"]*len(test_record_high_waveforms),return_dict=True)



In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

checkpoint_path = "assets/v2/15_stage1_True/checkpoint-2960/pytorch_model.bin"

model = StyleTransform(device=device)
state_dict = torch.load(checkpoint_path, map_location=device)
state_dict = {k[6:]: v for k, v in state_dict.items()}
model.load_state_dict(state_dict)
model.eval()  

model = model.to(device)

sample_idx = random.randint(0, len(test_dataset_y1) - 1) # 626 7 747 54 793 568 1077 856
print(sample_idx)

sample = test_dataset_y1[sample_idx]

input_waveform = sample['input_values'].unsqueeze(0).unsqueeze(0)
target_waveform = sample['labels'].numpy()  
text_emb = sample['text_emb']

with torch.no_grad():
    enhanced_waveform = model(input_waveform, text_emb).squeeze(0).squeeze(0).cpu().numpy()

print("Original Digital Audio:")
display(Audio(input_waveform.squeeze().numpy(), rate=44100)) 

print("Target Audio:")
display(Audio(target_waveform, rate=44100))

print("Output Audio:")
display(Audio(enhanced_waveform, rate=44100))


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

checkpoint_path = "assets/v2/15_stage1_True/checkpoint-2960/pytorch_model.bin"

model = StyleTransform(device=device)
state_dict = torch.load(checkpoint_path, map_location=device)
state_dict = {k[6:]: v for k, v in state_dict.items()}
model.load_state_dict(state_dict)
model.eval()  

model = model.to(device)

sample_idx = random.randint(0, len(test_dataset_y2) - 1) # 626 7 747 54 793 568 1077 856
print(sample_idx)

sample = test_dataset_y2[sample_idx]

input_waveform = sample['input_values'].unsqueeze(0).unsqueeze(0)
target_waveform = sample['labels'].numpy()  
text_emb = sample['text_emb']

with torch.no_grad():
    enhanced_waveform = model(input_waveform, text_emb).squeeze(0).squeeze(0).cpu().numpy()

print("Original Digital Audio:")
display(Audio(input_waveform.squeeze().numpy(), rate=44100)) 

print("Target Audio:")
display(Audio(target_waveform, rate=44100))

print("Output Audio:")
display(Audio(enhanced_waveform, rate=44100))


## Others


In [ ]:
import json

def read_json(file_path):
    with open(file_path, "r") as f:
        return json.load(f)

converter = read_json("converter.json")
print(converter)

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

def get_color(i):
    color_map = ['blue', 'green', 'red', 'purple', 'cyan', 'magenta', 'yellow', 'black']
    return color_map[(i-1) % len(color_map)]  

N = 6  

for i in range(1, N + 1):
    dir_path = f"assets/embeddings/y{i}"
    os.makedirs(dir_path, exist_ok=True)

    color_i = get_color(i)

    file_paths = [f'{dir_path}/y{i}.txt', 'assets/target.txt']
    device = converter[f'y{i}']
    labels = [f'{device}', 'Harman Target']
    plot_colors = [color_i, 'orange']

    plt.figure(figsize=(10, 6))

    for file_path, label, color in zip(file_paths, labels, plot_colors):
        if os.path.exists(file_path):
            df = pd.read_csv(file_path, sep="\t", header=None, names=["Frequency", "Level"])
            plt.plot(df["Frequency"], df["Level"], marker='o', linestyle='-', color=color, label=label)
    plt.title(f"Frequency Response Curve of {device} and Harman Standard Curve")
    plt.xlabel("Frequency (Hz)")
    plt.ylabel("Level (dB)")
    plt.legend()
    plt.savefig(f"{dir_path}/y{i}.jpg", format="jpg", dpi=300)
    plt.close()


In [ ]:
import torch
import os
import numpy as np
import lovelyplots
from sklearn.preprocessing import normalize
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

plt.style.use(['ipynb'])

device_folders = f"assets/embeddings/"

device_embeddings = []
device_labels = []

for device_idx, device_name in enumerate(sorted(os.listdir(device_folders))):
    device_path = os.path.join(device_folders, device_name)
    
    if not os.path.isdir(device_path):
        continue
    
    embeddings = []
    
    for file in sorted(os.listdir(device_path)):
        if file.endswith(".pt"):
            tensor = torch.load(os.path.join(device_path, file)) 
            embeddings.append(tensor.squeeze().detach().numpy())  
    
    if embeddings:
        device_embeddings.extend(embeddings)
        device_labels.extend([device_name] * len(embeddings))

device_embeddings = np.array(device_embeddings)
device_labels = np.array(device_labels)

device_embeddings = normalize(device_embeddings, norm="l2")

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(device_embeddings, device_labels, test_size=0.9, random_state=42, stratify=device_labels)

# Train SVM classifier
clf = SVC(kernel="linear", C=10, random_state=42)  
clf.fit(X_train, y_train)

# Predict and evaluate
y_pred = clf.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"SVM Classification Accuracy: {acc:.4f}")

# t-SNE Visualization
tsne = TSNE(n_components=2, perplexity=50, metric="cosine", random_state=40)
embeddings_2d = tsne.fit_transform(device_embeddings)

plt.figure(figsize=(10, 8))
for device_name in set(device_labels):
    indices = np.where(device_labels == device_name)[0]
    plt.scatter(embeddings_2d[indices, 0], embeddings_2d[indices, 1], label=device_name, alpha=0.7)

plt.legend()
plt.title("t-SNE Visualization of Device Embeddings")
plt.xlabel("t-SNE Component 1")
plt.ylabel("t-SNE Component 2")
plt.show()


In [ ]:
# import lovelyplots
import matplotlib.pyplot as plt
# plt.style.use(['ipynb'])
import numpy as np

# Data
training_files = [15, 30, 45, 60]
train_samples = [4728, 11820, 18912, 26004]
val_samples = [1182, 2955, 4728, 6501]
test_samples = [2955, 2955, 2955, 2955]
test_mae = [0.0023,0.0020, 0.0019, 0.0013]

# Bar width and positions
bar_width = 0.25
x = np.arange(len(training_files))

# Plot
fig, ax1 = plt.subplots(figsize=(12, 6))

# Bar plots for train, val, and test samples
ax1.bar(x - bar_width, train_samples, width=bar_width, color='skyblue', label='Train Samples')
ax1.bar(x, val_samples, width=bar_width, color='lightgreen', label='Val Samples')
ax1.bar(x + bar_width, test_samples, width=bar_width, color='salmon', label='Test Samples')

# Primary y-axis for samples
ax1.set_ylabel("Number of Samples", fontsize=12)
ax1.set_xlabel("Number of Training Files", fontsize=12)
ax1.set_xticks(x)
ax1.set_xticklabels(training_files, fontsize=10)
ax1.legend(loc="upper center", fontsize=10)
# ax1.grid(axis='y', linestyle='--', alpha=0.7)

# Secondary y-axis for MAE
ax2 = ax1.twinx()
ax2.plot(x, test_mae, color='black', marker='o', label='Test MAE', linewidth=2)
ax2.set_ylabel("Test MAE", fontsize=12)
ax2.legend(loc="upper right", fontsize=10)

# Add raw MAE values on the line markers
for i in range(len(x)):
    ax2.text(x[i], test_mae[i] + 0.00002, f"{test_mae[i]:.4f}", ha='center', fontsize=10, color='black')

# Title and layout
plt.title("Test MAE in Different Number of Training Configurations", fontsize=14)
plt.tight_layout()
plt.show()


In [78]:
!rm assets/embeddings/y1/*.pt